# Yaw Flip Validation: GPM

This notebook validates TAT-C against the [Global Precipitation Measurement (GPM) Core Observatory](https://gpm.nasa.gov/missions/GPM/core-observatory) across one of its yaw maneuvers. GPM flies a non-sun-synchronous orbit at 407 km altitude and 65 deg inclination, so the local time of its orbit precesses through the day in about 83 days (Test 1). To keep the Sun on the same side of the spacecraft, GPM turns 180 deg about its vertical axis (a yaw flip) about every 40 days, as the Sun crosses its orbit plane, alternately flying forward (orientation 0 deg) and backward (180 deg). It carries two instruments:

* the **Dual-frequency Precipitation Radar (DPR)**, whose Ku-band radar scans across track within about 17 deg of nadir in 49 beam positions (a swath of about 245 km);
* the **GPM Microwave Imager (GMI)**, a conically scanning radiometer whose antenna looks 48.5 deg from nadir over a sector of about 150 deg ahead of the spacecraft (when flying forward), covering a swath of about 885 km (validated in [Conical Scanner Validation: GPM GMI](ValidateConicalGMI.ipynb) for the forward orientation).

A yaw flip reverses the order of the radar's beams across the swath, and turns GMI's scan sector from ahead of the spacecraft to behind it. TAT-C represents the radar as a `PointedInstrument` in scan geometry and GMI as a `ConicalInstrument`, whose scan sector is centered at a fixed azimuth (`scan_center_azimuth`). This notebook asks:

1. **Orbit.** Does TAT-C's propagation of the TLE reproduce GPM's position, and the precession of its non-sun-synchronous orbit?
2. **Radar scan geometry.** What are the radar's scan angles and velocity frame, and how does the yaw flip change them?
3. **GMI scan geometry.** How does the yaw flip change GMI's scan sector?
4. **Observation times.** Does `collect_observations` predict when the radar and GMI observe points, before and after the flip?

## Reference Data

Two products from the [Goddard Earth Sciences Data and Information Services Center](https://disc.gsfc.nasa.gov/) (GES DISC), version 08, provide one netCDF file per orbit:

* **Ku-band radar profiles** (`GPM_2AKu`): for each scan (every 0.6 s), its time, the spacecraft's Earth-fixed position, velocity, and attitude, its orientation (0 or 180 deg), and the geodetic latitude and longitude of each of the 49 beam positions at the surface;
* **GMI brightness temperatures** (`GPM_1CGPMGMI`): for each scan of the low-frequency channels (S1, every 1.875 s), its time and orientation and the latitude and longitude of each of its 221 pixels.

The 232 orbits of 10 to 24 September 2026, across the yaw flip of 17 September, are reduced with HTTP range requests (read with `h5py`) to every fifth radar scan and every third GMI scan. Downloading requires a free [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account and the `earthaccess` package; the first run takes about an hour, and the reduced data are cached in a local `data/gpm` directory. GPM's orbit is taken from its TLE from [CelesTrak](https://celestrak.org/) (epoch 6 October 2026, 06:57 UTC).

In [1]:
import concurrent.futures
import io
from datetime import datetime, timedelta, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import requests

DATA_DIR = Path("data") / "gpm"
DATA_DIR.mkdir(parents=True, exist_ok=True)
START, END = "2026-09-10T00:00:00Z", "2026-09-24T23:59:59Z"
EPOCH = pd.Timestamp("2000-01-01", tz="UTC")
TLE = [
    "1 39574U 14009C   26279.28972394  .00005183  00000+0  11363-3 0  9995",
    "2 39574  64.9698 295.8138 0010766 269.3181  90.6726 15.45237411714979",
]

_sessions = []


def https_session():
    """An authenticated Earthdata HTTPS session, logging in on first use."""
    if not _sessions:
        import earthaccess

        earthaccess.login()
        _sessions.append(earthaccess.get_requests_https_session())
    return _sessions[0]


class RemoteFile(io.RawIOBase):
    """Read-only file object that downloads blocks of a remote file with HTTP range requests on demand."""

    def __init__(self, url, block_size=2**20, prefetch=1):
        self.block_size, self.blocks, self.position = block_size, {}, 0
        response = self._get(url, 0, prefetch * block_size)
        self.url, self.size = response.url, int(response.headers["Content-Range"].split("/")[1])
        self._store(0, response.content)

    def _get(self, url, start, end):
        for attempt in range(5):
            try:
                response = https_session().get(url, headers={"Range": f"bytes={start}-{end - 1}"}, timeout=300)
                response.raise_for_status()
                return response
            except requests.RequestException:
                if attempt == 4:
                    raise

    def _store(self, first, content):
        for i in range(0, len(content), self.block_size):
            self.blocks[first + i // self.block_size] = content[i : i + self.block_size]

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.position

    def seek(self, offset, whence=io.SEEK_SET):
        self.position = {io.SEEK_SET: 0, io.SEEK_CUR: self.position, io.SEEK_END: self.size}[whence] + offset
        return self.position

    def readinto(self, buffer):
        n = min(len(buffer), self.size - self.position)
        if n <= 0:
            return 0
        first, last = self.position // self.block_size, (self.position + n - 1) // self.block_size
        missing = [block for block in range(first, last + 1) if block not in self.blocks]
        if missing:
            end = min((missing[-1] + 1) * self.block_size, self.size)
            self._store(missing[0], self._get(self.url, missing[0] * self.block_size, end).content)
        data = b"".join(self.blocks[block] for block in range(first, last + 1))
        offset = self.position - first * self.block_size
        buffer[:n] = data[offset : offset + n]
        self.position += n
        return n


def scan_seconds(group):
    """Scan times as seconds since 2000-01-01 (UTC calendar fields)."""
    t = pd.to_datetime({"year": group["Year"][:], "month": group["Month"][:], "day": group["DayOfMonth"][:], "hour": group["Hour"][:], "minute": group["Minute"][:], "second": group["Second"][:]}) + pd.to_timedelta(group["MilliSecond"][:], unit="ms")
    return ((t - pd.Timestamp("2000-01-01")).dt.total_seconds()).to_numpy()


def reduce_ku(url):
    path = DATA_DIR / url.rsplit("/", 1)[1].replace(".nc", ".npz")
    if path.exists():
        return path
    with h5py.File(RemoteFile(url), "r") as f:
        g, k = f["FS"], slice(None, None, 5)
        np.savez_compressed(
            path,
            seconds=scan_seconds(g["ScanTime"])[k], sc_pos=g["navigation/scPos"][:][k], sc_vel=g["navigation/scVel"][:][k],
            yaw=g["navigation/scAttYawGeod"][:][k], roll=g["navigation/scAttRollGeod"][:][k], pitch=g["navigation/scAttPitchGeod"][:][k],
            orientation=g["scanStatus/SCorientation"][:][k], latitude=g["Latitude"][:][k], longitude=g["Longitude"][:][k],
            zenith=g["PRE/localZenithAngle"][:][k], elevation=g["PRE/elevation"][:][k],
        )
    return path


def reduce_gmi(url):
    path = DATA_DIR / url.rsplit("/", 1)[1].replace(".nc", ".npz")
    if path.exists():
        return path
    with h5py.File(RemoteFile(url), "r") as f:
        g, k = f["S1"], slice(None, None, 3)
        np.savez_compressed(
            path,
            seconds=scan_seconds(g["ScanTime"])[k], orientation=g["SCstatus/SCorientation"][:][k],
            sc_latitude=g["SCstatus/SClatitude"][:][k], sc_longitude=g["SCstatus/SClongitude"][:][k], sc_altitude=g["SCstatus/SCaltitude"][:][k],
            latitude=g["Latitude"][:][k], longitude=g["Longitude"][:][k], incidence=g["incidenceAngle"][:][k][..., 0],
        )
    return path


if not any(DATA_DIR.glob("2A.GPM.Ku.*.npz")):
    import earthaccess

    jobs = []
    for short_name, reducer in (("GPM_2AKu", reduce_ku), ("GPM_1CGPMGMI", reduce_gmi)):
        results = earthaccess.search_data(short_name=short_name, temporal=(START, END))
        jobs += [(reducer, next(link["URL"] for link in r["umm"]["RelatedUrls"] if link["URL"].startswith("https") and link["URL"].endswith(".nc"))) for r in results]
    with concurrent.futures.ThreadPoolExecutor(6) as pool:
        list(pool.map(lambda job: job[0](job[1]), jobs))


def load(pattern):
    out = {}
    for path in sorted(DATA_DIR.glob(pattern)):
        with np.load(path) as d:
            record = {key: d[key] for key in d.files}
        record["name"], record["orbit"] = path.stem, int(path.name.split(".")[5])
        record["times"] = EPOCH + pd.to_timedelta(record["seconds"], unit="s")
        out[record["orbit"]] = record
    return out


ku, gmi = load("2A.GPM.Ku.*.npz"), load("1C.GPM.GMI.*.npz")
orientation = pd.concat([pd.DataFrame({"time": r["times"], "orientation": r["orientation"]}) for r in ku.values()]).sort_values("time").reset_index(drop=True)
missing = int((~orientation.orientation.isin([0, 180])).sum())
orientation = orientation[orientation.orientation.isin([0, 180])].reset_index(drop=True)
changes = orientation[orientation.orientation.diff().fillna(0) != 0]
FLIP = changes.time.iloc[0] if len(changes) else None
pd.Series(
    {
        "radar orbits": len(ku), "GMI orbits": len(gmi),
        "first scan": orientation.time.min(), "last scan": orientation.time.max(),
        "orientations": ", ".join(f"{o} deg ({n} scans)" for o, n in orientation.orientation.value_counts().sort_index().items()) + f"; {missing} sampled scans without orientation",
        "orientation change": FLIP,
    }
)

radar orbits                                                        232
GMI orbits                                                          232
first scan                          2026-09-09 23:51:53.151000023+00:00
last scan                           2026-09-25 00:17:05.052000046+00:00
orientations          0 deg (183218 scans), 180 deg (187223 scans); ...
orientation change                  2026-09-17 14:06:02.935000062+00:00
dtype: object

## Setup

In [2]:
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.spatial import cKDTree
from skyfield.api import wgs84
from skyfield.framelib import itrs
from skyfield.positionlib import Geocentric
from skyfield.units import Distance, Velocity

from tatc.analysis import collect_observations
from tatc.constants import timescale
from tatc.generation import generate_points_fibonacci_lattice
from tatc.schemas import ConicalInstrument, GeneralPerturbationsOrbit, Point, PointedInstrument, Satellite
from tatc.utils import VelocityFrame, ViewGeometry, compute_cone_and_azimuth, compute_view_angles, geodesic_distance

orbit = GeneralPerturbationsOrbit.from_tle(TLE)
BLUE, ORANGE, AQUA, GRAY, INK, PURPLE = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984", "#0b0b0b", "#8e5bd8"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6})


def times_of(values):
    return timescale.from_datetimes([t.to_pydatetime() for t in pd.DatetimeIndex(values)])


def ephemeris_track(record, k):
    """GPM's reported (Earth-fixed) position and velocity at radar scans k, as an inertial orbit track."""
    return Geocentric.from_time_and_frame_vectors(times_of(record["times"][k]), itrs, Distance(m=record["sc_pos"][k].T.astype(float)), Velocity(km_per_s=record["sc_vel"][k].T.astype(float) / 1e3))

## Test 1: Orbit

The TLE is propagated (SGP4) to every 50th sampled radar scan and compared with the reported positions in the along-track, cross-track, and radial directions, by day. The local time of the ascending node is found from the reported positions and from the TLE.

In [3]:
records = []
for record in ku.values():
    k = np.arange(0, len(record["seconds"]), 50)
    reported, velocity = record["sc_pos"][k].T.astype(float), record["sc_vel"][k].T.astype(float)
    tle_xyz = orbit.get_orbit_track(list(record["times"][k].to_pydatetime())).frame_xyz(itrs).m
    radial = reported / np.linalg.norm(reported, axis=0)
    cross = np.cross(reported, velocity, axis=0)
    cross /= np.linalg.norm(cross, axis=0)
    along = np.cross(cross, radial, axis=0)
    difference = (tle_xyz - reported) / 1e3
    records.append(pd.DataFrame({"time": record["times"][k], "along-track (km)": np.sum(difference * along, axis=0), "cross-track (km)": np.sum(difference * cross, axis=0), "radial (km)": np.sum(difference * radial, axis=0)}))
errors = pd.concat(records)
errors["day"] = errors.time.dt.strftime("%m-%d")
errors["days before TLE epoch"] = ((pd.Timestamp(orbit.get_epoch()) - errors.time).dt.total_seconds() / 86400).round(0)
nodes = []
for record in ku.values():
    z = record["sc_pos"][:, 2]
    for i in np.nonzero((z[:-1] < 0) & (z[1:] >= 0))[0]:
        f = -z[i] / (z[i + 1] - z[i])
        time = record["times"][i] + (record["times"][i + 1] - record["times"][i]) * f
        x, y = (record["sc_pos"][i, j] + f * (record["sc_pos"][i + 1, j] - record["sc_pos"][i, j]) for j in (0, 1))
        nodes.append({"time": time, "local time (h)": ((time - time.normalize()).total_seconds() / 3600 + np.degrees(np.arctan2(y, x)) / 15) % 24})
nodes = pd.DataFrame(nodes)


def tle_node_local_time(time):
    seconds = np.arange(-3000, 3001, 10)
    xyz = orbit.get_orbit_track([time.to_pydatetime() + timedelta(seconds=float(s)) for s in seconds]).frame_xyz(itrs).m
    k = np.nonzero((xyz[2, :-1] < 0) & (xyz[2, 1:] >= 0))[0]
    k = k[np.argmin(np.abs(seconds[k]))]
    f = -xyz[2, k] / (xyz[2, k + 1] - xyz[2, k])
    crossing = time + pd.Timedelta(seconds=float(seconds[k] + 10 * f))
    lon = np.degrees(np.arctan2(xyz[1, k] + f * (xyz[1, k + 1] - xyz[1, k]), xyz[0, k] + f * (xyz[0, k + 1] - xyz[0, k])))
    return ((crossing - crossing.normalize()).total_seconds() / 3600 + lon / 15) % 24


nodes["TAT-C local time (h)"] = [tle_node_local_time(t) for t in nodes.time]
nodes["TAT-C minus reported (min)"] = 60 * ((nodes["TAT-C local time (h)"] - nodes["local time (h)"] + 12) % 24 - 12)
rate = 60 * np.polyfit((nodes.time - nodes.time.iloc[0]).dt.total_seconds() / 86400, np.unwrap(nodes["local time (h)"] * np.pi / 12) * 12 / np.pi, 1)[0]
print(f"ascending node local time: {nodes['local time (h)'].iloc[0]:.2f} h on {nodes.time.iloc[0]:%d %b} to {nodes['local time (h)'].iloc[-1]:.2f} h on {nodes.time.iloc[-1]:%d %b} ({rate:.1f} min/day); TAT-C minus reported within {nodes['TAT-C minus reported (min)'].abs().max():.2f} min")
errors.groupby("days before TLE epoch")[["along-track (km)", "cross-track (km)", "radial (km)"]].median().round(2).T

C:\Users\pgrogan\AppData\Local\Temp\ipykernel_40436\741476047.py:28: UserWarning: Discarding nonzero nanoseconds in conversion.
  xyz = orbit.get_orbit_track([time.to_pydatetime() + timedelta(seconds=float(s)) for s in seconds]).frame_xyz(itrs).m


ascending node local time: 2.32 h on 10 Sep to 22.00 h on 24 Sep (-17.3 min/day); TAT-C minus reported within 0.15 min


days before TLE epoch,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0
along-track (km),30.76,26.36,17.23,6.40,-6.31,-20.49,-36.43,-54.15,-73.69,-95.92,-122.03,-151.99,-185.04,-221.77,-260.82,-298.58
cross-track (km),-0.08,-0.06,-0.06,-0.01,-0.03,-0.13,0.13,-0.22,0.25,-0.39,0.31,-0.25,0.09,-0.09,-0.04,0.15
radial (km),0.17,-0.15,-0.07,-0.11,-0.07,-0.11,-0.28,-0.27,-0.66,-0.75,-1.35,-1.88,-2.78,-3.84,-5.29,-6.94


GPM flipped from flying backward (orientation 180 deg) to forward (0 deg) at 14:06 UTC on 17 September.

The TLE reproduces the precession of GPM's orbit: the local time of the ascending node moves 17.3 minutes earlier per day (from 2.3 h on 10 September to 22.0 h on 24 September, a full cycle in about 83 days, so that the Sun crosses the orbit plane about every 42 days, the interval of the yaw flips), and the TLE's local times are within 0.15 minutes of the reported ones. GPM's low orbit, however, decays under atmospheric drag (and is raised by maneuvers), so the TLE's along-track position departs from the reported one away from its epoch: by 31 km 11 days before it, passing through zero 14 to 15 days before it, and growing roughly quadratically to 150 km at 22 days and 300 km at 26 days, while the cross-track and radial differences stay within 0.4 and 7 km. Over the six days around the flip (16 to 22 days before the TLE epoch), this shifts the predicted observation times by 2 to 19 s (Test 4).

## Test 2: Radar Scan Geometry

For sampled radar scans, the view angles of each beam position are computed from GPM's reported position and velocity with `compute_view_angles` (roll positive left, pitch positive forward), in both velocity frames, separately for each orientation.

In [4]:
def beam_angles(record, step=60):
    rows = []
    for s in range(10, len(record["seconds"]), step):
        track = ephemeris_track(record, [s])[0]
        for frame in VelocityFrame:
            angles = np.array([compute_view_angles(track, wgs84.latlon(float(record["latitude"][s, j]), float(record["longitude"][s, j])), velocity_frame=frame) for j in range(49)], dtype=float)
            rows.append({"orientation": int(record["orientation"][s]), "frame": frame.value, "beam 1 roll (deg)": angles[0, 0], "beam 25 roll (deg)": angles[24, 0], "beam 49 roll (deg)": angles[48, 0], "pitch, mean (deg)": angles[:, 1].mean(), "pitch trend (deg per deg)": np.polyfit(angles[:, 0], angles[:, 1], 1)[0]})
    return rows


beams = pd.DataFrame([row for rows in Parallel(n_jobs=-1)(delayed(beam_angles)(r) for r in list(ku.values())[::6]) for row in rows])
beams = beams[beams.orientation.isin([0, 180])]
beams.groupby(["orientation", "frame"]).agg(["median", "std"]).round(4).T

orientation                              0                    180         
frame                            earth_fixed inertial earth_fixed inertial
beam 1 roll (deg)         median     17.0427  17.0539    -17.0432 -17.0547
                          std         0.0148   0.0113      0.0147   0.0113
beam 25 roll (deg)        median      0.0548   0.0548     -0.0548  -0.0549
                          std         0.0041   0.0003      0.0041   0.0004
beam 49 roll (deg)        median    -16.9411 -16.9518     16.9413  16.9524
                          std         0.0146   0.0113      0.0148   0.0113
pitch, mean (deg)         median     -0.1521  -0.1521      0.1522   0.1522
                          std         0.0022   0.0003      0.0022   0.0004
pitch trend (deg per deg) median     -0.0143  -0.0167      0.0184   0.0166
                          std         0.0422   0.0002      0.0421   0.0002

The radar's 49 beam positions span -17.05 to +16.95 deg of roll (0.708 deg apart), with the first beam on the right of the direction of motion when GPM flies backward and on the left when it flies forward: the yaw flip reverses the order of the beams. The beams also point 0.15 deg along track toward the spacecraft's forward axis, so behind the satellite when it flies backward and ahead of it when it flies forward. The scans are perpendicular to the inertial velocity (GPM is not yaw-steered): in the inertial frame, the along-track angle of the beams varies across the scan by the same amount at all times (0.017 deg per degree of roll, from the scan's duration), whereas in the Earth-fixed frame its variation changes around the orbit (standard deviation 0.04 deg per degree).

A radar `PointedInstrument` is defined for each orientation from these values: 49 pixels across track spanning 34.7 deg (49 beams of 0.708 deg), the orientation's pitch angle (0.15 deg forward or aft), the inertial velocity frame, and scan geometry. TAT-C's cross-track pixel index 0 is on the right of the direction of motion, so beam 1 is pixel 0 in one orientation and pixel 48 in the other. The positions of beams 1, 25, and 49 are compared with the reported ones (using the reported orbit), with each orientation's instrument and pixel order, and with the first orientation's throughout.

In [5]:
inertial = beams[beams.frame == "inertial"].groupby("orientation").median(numeric_only=True)
radar = {
    o: PointedInstrument(name="Ku", field_of_regard=36, cross_track_field_of_view=49 * 0.708, along_track_field_of_view=0.7, pitch_angle=round(float(inertial.loc[o, "pitch, mean (deg)"]), 2), cross_track_pixels=49, is_rectangular=True, view_geometry=ViewGeometry.SCAN, velocity_frame=VelocityFrame.INERTIAL)
    for o in (0, 180)
}
first_orientation = int(orientation.orientation.iloc[0])
beam1_pixel = {o: (0 if inertial.loc[o, "beam 1 roll (deg)"] < 0 else 48) for o in (0, 180)}


def beam_errors(record, step=40):
    rows = []
    for s in range(5, len(record["seconds"]), step):
        track = ephemeris_track(record, [s])[0]
        o = int(record["orientation"][s])
        if o not in (0, 180):
            continue
        for beam in (1, 25, 49):
            lat, lon = float(record["latitude"][s, beam - 1]), float(record["longitude"][s, beam - 1])
            for label, model in (("orientation's instrument", o), ("first orientation's instrument", first_orientation)):
                pixel = (beam - 1) if beam1_pixel[model] == 0 else (49 - beam)
                position = radar[model].compute_projected_pixel_position(track, pixel, 0)
                rows.append({"orientation": o, "beam": beam, "model": label, "distance (km)": geodesic_distance(lon, lat, position.longitude.degrees, position.latitude.degrees) / 1e3})
    return rows


beam_table = pd.DataFrame([row for rows in Parallel(n_jobs=-1)(delayed(beam_errors)(r) for r in list(ku.values())[::3]) for row in rows])
print("beam 1 is TAT-C pixel", beam1_pixel, "; pitch angles (deg):", {o: r.pitch_angle for o, r in radar.items()})
beam_table.groupby(["model", "orientation", "beam"])["distance (km)"].median().unstack("beam").round(2)

beam 1 is TAT-C pixel {0: 48, 180: 0} ; pitch angles (deg): {0: -0.15, 180: 0.15}


beam                                            1     25      49
model                          orientation                      
first orientation's instrument 0            270.67  1.94  269.74
                               180            1.42  0.59    3.18
orientation's instrument       0              3.22  0.59    1.37
                               180            1.42  0.59    3.18

With each orientation's instrument and pixel order, the beam positions are within 0.6 km (median) at nadir and 1.4 to 3.2 km at the swath edges (largest for the beam scanned first). With the first orientation's instrument after the flip, the beams at the edges are 270 km away, on the other side of the swath; nadir positions are unaffected. Since the swath is symmetric, the radar's coverage does not depend on the orientation, but its pixel order does.

## Test 3: GMI Scan Geometry

For sampled GMI scans, the cone angle (from nadir) and scan azimuth (from the along-track direction, positive left, in the inertial velocity frame) of each pixel are computed with `compute_cone_and_azimuth` from GPM's position and velocity, interpolated from the radar's navigation at the GMI scan time.

In [6]:
def gmi_angles(number, step=150):
    g, k = gmi[number], ku.get(number)
    if k is None:
        return []
    rows = []
    for s in range(20, len(g["seconds"]), step):
        i = np.searchsorted(k["seconds"], g["seconds"][s])
        if i == 0 or i >= len(k["seconds"]):
            continue
        f = (g["seconds"][s] - k["seconds"][i - 1]) / (k["seconds"][i] - k["seconds"][i - 1])
        pos = (1 - f) * k["sc_pos"][i - 1] + f * k["sc_pos"][i]
        vel = (1 - f) * k["sc_vel"][i - 1] + f * k["sc_vel"][i]
        track = Geocentric.from_time_and_frame_vectors(times_of([g["times"][s]])[0], itrs, Distance(m=pos.astype(float)), Velocity(km_per_s=vel.astype(float) / 1e3))
        angles = np.array([compute_cone_and_azimuth(track, wgs84.latlon(float(g["latitude"][s, j]), float(g["longitude"][s, j])), VelocityFrame.INERTIAL) for j in range(0, 221, 10)], dtype=float)
        azimuth = np.degrees(np.unwrap(np.radians(angles[:, 1])))
        rows.append({"orientation": int(g["orientation"][s]), "cone angle (deg)": np.median(angles[:, 0]), "first pixel azimuth (deg)": azimuth[0], "last pixel azimuth (deg)": azimuth[-1], "sector center (deg)": (((azimuth[0] + azimuth[-1]) / 2 + 180) % 360) - 180, "sector half width (deg)": abs(azimuth[-1] - azimuth[0]) / 2})
    return rows


gmi_table = pd.DataFrame([row for rows in Parallel(n_jobs=-1)(delayed(gmi_angles)(n) for n in list(gmi)[::4]) for row in rows])
gmi_table = gmi_table[gmi_table.orientation.isin([0, 180])]
sector = gmi_table.groupby("orientation").median()
sector.round(2).T

orientation,0,180
cone angle (deg),48.58,48.32
first pixel azimuth (deg),-75.95,103.90
last pixel azimuth (deg),75.39,256.70
sector center (deg),-0.28,-179.70
sector half width (deg),75.67,76.40


GMI's cone angle is 48.3 to 48.6 deg in both orientations, and its scan sector spans about 76 deg on either side of the spacecraft's forward axis: centered at -0.3 deg (ahead of the satellite) when GPM flies forward and at -179.7 deg (behind it) when it flies backward. A `ConicalInstrument` represents each orientation with its scan center azimuth; the yaw flip changes the azimuth by 180 deg.

## Test 4: Observation Times

The points of interest are those of a Fibonacci lattice with a typical spacing of 500 km, within 65 deg of the equator. For each point and pass, the radar's and GMI's observation times are the times of their scans through the point, interpolated between sampled scans from the nearest sampled pixel (within 15 km for the radar and 30 km for GMI, excluding the outermost pixels), over the three days before and after the yaw flip.

`collect_observations` predicts the observations of the radar `PointedInstrument` of each orientation (analyzed separately before and after the flip) and of a GMI `ConicalInstrument` (with the cone angle and scan sector of Test 3) with the TLE orbit. Two GMI models are compared: with the scan sector of each orientation (analyzed separately before and after the flip), and with the forward sector throughout. Since the TLE's along-track error shifts all predicted times of a pass equally (Test 1), GMI's observation time is also compared relative to the radar's on the same pass.

In [7]:
WINDOW = (FLIP - pd.Timedelta(days=3), FLIP + pd.Timedelta(days=3))
lattice = generate_points_fibonacci_lattice(500e3)
points = [Point(id=i, latitude=r.geometry.y, longitude=r.geometry.x) for i, r in enumerate(lattice.itertuples()) if abs(r.geometry.y) < 65]


def reference_times(records, radius, sample):
    xyz, index = [], []
    for number, r in records.items():
        if r["times"][-1] < WINDOW[0] or r["times"][0] > WINDOW[1]:
            continue
        rows, cols = np.nonzero(np.isfinite(r["latitude"]) & (np.abs(r["latitude"]) <= 90))
        cols_ok = (cols > 0) & (cols < r["latitude"].shape[1] - 1) & (cols % sample == 0)
        rows, cols = rows[cols_ok], cols[cols_ok]
        xyz.append(wgs84.latlon(r["latitude"][rows, cols], r["longitude"][rows, cols]).itrs_xyz.m.T)
        index.append(np.column_stack([np.full(len(rows), number), rows, cols]))
    xyz, index = np.concatenate(xyz), np.concatenate(index)
    tree = cKDTree(xyz)
    out = []
    for point in points:
        p = np.array(wgs84.latlon(point.latitude, point.longitude).itrs_xyz.m)
        nearby = tree.query_ball_point(p, radius)
        if not nearby:
            continue
        candidates = pd.DataFrame(index[nearby], columns=["orbit", "row", "column"]).assign(distance=np.linalg.norm(xyz[nearby] - p, axis=1))
        candidates["time"] = [records[o]["times"][r] for o, r in zip(candidates.orbit, candidates.row)]
        candidates = candidates.sort_values("time")
        for _, group in candidates.groupby((candidates.time.diff().dt.total_seconds() > 600).cumsum()):
            best = group.loc[group.distance.idxmin()]
            r, row, col = records[int(best.orbit)], int(best.row), int(best.column)
            r0, r1 = (row, row + 1) if row + 1 < len(r["seconds"]) else (row - 1, row)
            x0, x1 = (np.array(wgs84.latlon(float(r["latitude"][k, col]), float(r["longitude"][k, col])).itrs_xyz.m) for k in (r0, r1))
            fraction = float(np.dot(p - x0, x1 - x0) / np.dot(x1 - x0, x1 - x0))
            time = r["times"][r0] + (r["times"][r1] - r["times"][r0]) * fraction
            if WINDOW[0] <= time <= WINDOW[1]:
                out.append({"point_id": point.id, "time": time})
    return pd.DataFrame(out)


reference = {"radar": reference_times(ku, 15e3, 1), "GMI": reference_times(gmi, 30e3, 2)}
cone = float(sector["cone angle (deg)"].median())
forward = int(sector["sector center (deg)"].abs().idxmin())
half_width = float(sector["sector half width (deg)"].median())


def gmi_instrument(center):
    return ConicalInstrument(name="GMI", cone_angle=cone, scan_center_azimuth=center, scan_half_width=half_width, along_track_field_of_view=1, velocity_frame=VelocityFrame.INERTIAL)


periods = [(WINDOW[0], FLIP, int(orientation[orientation.time < FLIP].orientation.mode()[0])), (FLIP, WINDOW[1], int(orientation[orientation.time >= FLIP].orientation.mode()[0]))]
models = {
    "radar": [(radar[o], start, end) for start, end, o in periods],
    "GMI, sector by orientation": [(gmi_instrument(float(sector.loc[o, "sector center (deg)"])), start, end) for start, end, o in periods],
    "GMI, forward sector": [(gmi_instrument(float(sector.loc[forward, "sector center (deg)"])), start, end) for start, end, _ in periods],
}


def observe(point, instrument, start, end):
    return collect_observations(point, Satellite(name="GPM", orbit=orbit, instruments=[instrument]), start.to_pydatetime(), end.to_pydatetime()).assign(point_id=point.id)


predicted = {}
for label, runs in models.items():
    frames = Parallel(n_jobs=-1)(delayed(observe)(point, instrument, start, end) for instrument, start, end in runs for point in points)
    predicted[label] = pd.concat([f for f in frames if not f.empty])[["point_id", "epoch"]]


def match(observed, model):
    merged = pd.merge_asof(observed.sort_values("time"), predicted[model].rename(columns={"epoch": "predicted"}).sort_values("predicted"), left_on="time", right_on="predicted", by="point_id", direction="nearest", tolerance=pd.Timedelta(minutes=5))
    merged["difference (s)"] = (merged.predicted - merged.time).dt.total_seconds()
    merged["period"] = np.where(merged.time < FLIP, "before flip", "after flip")
    return merged


matched = {label: match(reference["radar" if label == "radar" else "GMI"], label) for label in models}
rows4 = {}
for label, merged in matched.items():
    for period, group in merged.groupby("period", sort=False):
        rows4[(label, period)] = {"observations": len(group), "matched (%)": 100 * group.predicted.notna().mean(), "time difference, median (s)": group["difference (s)"].median(), "time difference, 5th to 95th percentile (s)": f"{group['difference (s)'].quantile(0.05):.1f} to {group['difference (s)'].quantile(0.95):.1f}"}
pd.DataFrame(rows4).T

observations matched (%)  \
radar                      before flip         2521   96.231654   
                           after flip          2539   96.770382   
GMI, sector by orientation before flip         9158   95.370168   
                           after flip          9163   95.307214   
GMI, forward sector        before flip         9158   95.217296   
                           after flip          9163   95.307214   

                                       time difference, median (s)  \
radar                      before flip                   13.595436   
                           after flip                     5.420916   
GMI, sector by orientation before flip                   14.028344   
                           after flip                      5.88665   
GMI, forward sector        before flip                 -117.309244   
                           after flip                      5.88665   

                                       time difference, 5th to 95th percentile (s)  
radar                      before flip                                 9.5 to 18.6  
                           after flip                                   2.4 to 8.7  
GMI, sector by orientation before flip                                 9.5 to 19.5  
                           after flip                                   2.5 to 9.6  
GMI, forward sector        before flip                             -137.8 to -44.1  
                           after flip                                   2.5 to 9.6

In [8]:
# GMI's observation time relative to the radar's on the same pass
pairs = pd.merge_asof(matched["radar"].dropna(subset=["predicted"]).sort_values("time"), reference["GMI"].rename(columns={"time": "gmi_time"}).sort_values("gmi_time"), left_on="time", right_on="gmi_time", by="point_id", direction="nearest", tolerance=pd.Timedelta(minutes=5)).dropna(subset=["gmi_time"])
rows4b = {}
for label in ("GMI, sector by orientation", "GMI, forward sector"):
    p = pd.merge_asof(pairs.sort_values("gmi_time"), predicted[label].rename(columns={"epoch": "gmi_predicted"}).sort_values("gmi_predicted"), left_on="gmi_time", right_on="gmi_predicted", by="point_id", direction="nearest", tolerance=pd.Timedelta(minutes=5)).dropna(subset=["gmi_predicted"])
    p["reported (s)"] = (p.gmi_time - p.time).dt.total_seconds()
    p["TAT-C (s)"] = (p.gmi_predicted - p.predicted).dt.total_seconds()
    p["TAT-C minus reported (s)"] = p["TAT-C (s)"] - p["reported (s)"]
    for period, group in p.groupby("period", sort=False):
        rows4b[(label, period)] = {"passes": len(group), "GMI minus radar, reported, median (s)": group["reported (s)"].median(), "GMI minus radar, TAT-C, median (s)": group["TAT-C (s)"].median(), "TAT-C minus reported, median (s)": group["TAT-C minus reported (s)"].median(), "TAT-C minus reported, 95th percentile of magnitude (s)": group["TAT-C minus reported (s)"].abs().quantile(0.95)}
pd.DataFrame(rows4b).T.round(2)

passes  \
GMI, sector by orientation before flip  2423.0   
                           after flip   2453.0   
GMI, forward sector        before flip  2420.0   
                           after flip   2454.0   

                                        GMI minus radar, reported, median (s)  \
GMI, sector by orientation before flip                                  73.86   
                           after flip                                  -74.76   
GMI, forward sector        before flip                                  73.86   
                           after flip                                  -74.76   

                                        GMI minus radar, TAT-C, median (s)  \
GMI, sector by orientation before flip                               74.26   
                           after flip                               -74.24   
GMI, forward sector        before flip                              -73.93   
                           after flip                               -74.24   

                                        TAT-C minus reported, median (s)  \
GMI, sector by orientation before flip                              0.31   
                           after flip                               0.57   
GMI, forward sector        before flip                           -147.68   
                           after flip                               0.57   

                                        TAT-C minus reported, 95th percentile of magnitude (s)  
GMI, sector by orientation before flip                                               1.22       
                           after flip                                                1.06       
GMI, forward sector        before flip                                             151.83       
                           after flip                                                1.06

The absolute observation times are shifted by the TLE's along-track error (Test 1): the radar's predicted observations are 13.6 s late (median) before the flip and 5.4 s late after it, and GMI's (with the sector of each orientation) 14.0 and 5.9 s late; 96% of the radar's and 95% of GMI's observations are matched by predicted ones. Relative to the radar on the same pass, which removes the orbit error, GMI observes points 73.9 s after the radar before the flip, when its sector is behind the satellite, and 74.8 s before the radar after the flip, when its sector is ahead. With the sector of each orientation, TAT-C reproduces these times within 0.6 s (median) and 1.2 s (95th percentile). With the forward sector throughout, GMI's predicted observations before the flip are 148 s too early.

## Summary

| Test | Result |
| --- | --- |
| 1. Orbit (TLE) | node local time precession (17.3 minutes per day) within 0.15 minutes; along-track error from drag of up to 31 km within 11 to 17 days of the TLE epoch, growing to 300 km at 26 days |
| 2. Radar scan geometry (`ViewGeometry.SCAN`, `VelocityFrame.INERTIAL`) | 49 beams over -17 to +17 deg, perpendicular to the inertial velocity; the yaw flip reverses the beam order and the sign of a 0.15 deg along-track offset; beams within 0.6 to 3.2 km with each orientation's instrument, 270 km at the edges with the other orientation's |
| 3. GMI scan geometry (`ConicalInstrument`) | 48.5 deg cone; scan sector centered ahead (-0.3 deg) or behind (-179.7 deg) the satellite, depending on the orientation |
| 4. Observation times (`collect_observations`) | GMI relative to the radar within 0.6 s (median) with each orientation's sector; 148 s off with the forward sector before the flip; absolute times shifted by the TLE's along-track error |

**Yaw flips.** TAT-C represents each of GPM's orientations: the radar as a `PointedInstrument` with the orientation's pitch angle and pixel order, and GMI as a `ConicalInstrument` with the orientation's scan center azimuth. It does not model the flips themselves, so an analysis across a flip must be split at the time of the flip, with the instruments of each orientation; an attitude schedule (a time-dependent yaw of 0 or 180 deg) would represent it in a single analysis.

**Low orbits.** GPM's along-track position departs from its TLE by tens to hundreds of kilometers over weeks, because drag and orbit maintenance change its orbit; analyses of observation times should use TLEs close to the analysis period.